In [1]:
import pandas as pd
import plotly.express as px
import pandas as pd
import numpy as np

pd.set_option("display.max_columns", None)


In [2]:
# Lendo os dados para prosseguir para a limpeza e análise
df = pd.read_parquet('dados_taxi.parquet', engine='fastparquet')

df.head()


,VendorID,tpep_pickup_datetime,tpep_dropoff_datetime,passenger_count,trip_distance,RatecodeID,store_and_fwd_flag,PULocationID,DOLocationID,payment_type,fare_amount,extra,mta_tax,tip_amount,tolls_amount,improvement_surcharge,total_amount,congestion_surcharge,Airport_fee,cbd_congestion_fee
0,2,2026-01-01 00:54:04,2026-01-01 00:59:37,1.0,0.97,1.0,N,239,238,1,7.2,1.00,0.5,3.66,0.0,1.0,15.86,2.5,0.0,0.00
1,1,2026-01-01 00:34:04,2026-01-01 00:39:47,0.0,0.90,1.0,N,163,162,2,7.9,4.25,0.5,0.00,0.0,1.0,13.65,2.5,0.0,0.75
2,1,2026-01-01 00:57:06,2026-01-01 01:05:59,0.0,1.40,1.0,N,43,237,1,10.7,4.25,0.5,2.50,0.0,1.0,18.95,2.5,0.0,0.75
3,2,2026-01-01 00:15:22,2026-01-01 00:58:10,4.0,5.58,1.0,N,142,209,1,38.7,1.00,0.5,11.11,0.0,1.0,55.56,2.5,0.0,0.75
4,2,2026-01-01 00:27:13,2026-01-01 00:40:43,0.0,2.16,1.0,N,88,144,1,13.5,1.00,0.5,3.85,0.0,1.0,23.10,2.5,0.0,0.75


In [3]:
df['tpep_pickup_datetime'] = pd.to_datetime(df['tpep_pickup_datetime'])
df['tpep_dropoff_datetime'] = pd.to_datetime(df['tpep_dropoff_datetime'])

df_clean = df[(df['trip_distance'] > 0) & (df['fare_amount'] > 0)].copy()

df_clean = df_clean[df_clean['payment_type'].isin([1, 2])]

df_clean = df_clean.dropna(subset=['PULocationID', 'DOLocationID', 'trip_distance'])

In [4]:
df_clean['trip_duration_min'] = (df_clean['tpep_dropoff_datetime'] - df_clean['tpep_pickup_datetime']).dt.total_seconds() / 60

df_clean = df_clean[(df_clean['trip_duration_min'] > 1) & (df_clean['trip_duration_min'] < 180)]

df_clean['pickup_hour'] = df_clean['tpep_pickup_datetime'].dt.hour
df_clean['pickup_day_of_week'] = df_clean['tpep_pickup_datetime'].dt.day_name()

df_clean['dinheiro_recebido'] = df_clean['fare_amount'] + df_clean['tip_amount']

df_clean['dolares_por_hora'] = (df_clean['dinheiro_recebido'] / df_clean['trip_duration_min']) * 60

df_clean = df_clean[(df_clean['dolares_por_hora'] >= 10) & (df_clean['dolares_por_hora'] <= 200)]

df_zones = pd.read_csv('taxi_zone_lookup.csv')

df_clean = df_clean.merge(df_zones[['LocationID', 'Borough', 'Zone']], 
                          left_on='PULocationID', 
                          right_on='LocationID', 
                          how='left')

FileNotFoundError: [Errno 2] No such file or directory: 'taxi_zone_lookup.csv'

In [ ]:
url_zonas = 'https://d37ci6vzurychx.cloudfront.net/misc/taxi_zone_lookup.csv'
df_zonas = pd.read_csv(url_zonas)

horarios_rentaveis = df_clean.groupby('pickup_hour')['dolares_por_hora'].mean().sort_values(ascending=False).head(15)

print("--- TOP 15 HORÁRIOS QUE MAIS RENDEM (US$/Hora) ---")
for hora, valor in horarios_rentaveis.items():
    print(f"{hora}h: US$ {valor:.2f} / hora")
print("\n")

locais_rentaveis = df_clean.groupby('PULocationID')['dolares_por_hora'].mean().reset_index()

locais_com_nome = pd.merge(locais_rentaveis, df_zonas, left_on='PULocationID', right_on='LocationID', how='left')

top_locais = locais_com_nome.sort_values(by='dolares_por_hora', ascending=False).head(15)

print("--- TOP 15 LOCAIS DE EMBARQUE MAIS RENTÁVEIS (US$/Hora) ---")
for index, row in top_locais.iterrows():
    bairro = row['Zone']
    distrito = row['Borough']
    lucro = round(row['dolares_por_hora'], 2)
    print(f"{bairro} ({distrito}): US$ {lucro}/hora")

--- TOP 15 HORÁRIOS QUE MAIS RENDEM (US$/Hora) ---
4h: US$ 105.21 / hora
3h: US$ 103.12 / hora
0h: US$ 102.24 / hora
5h: US$ 101.56 / hora
1h: US$ 100.70 / hora
2h: US$ 100.60 / hora
23h: US$ 100.44 / hora
22h: US$ 96.55 / hora
21h: US$ 94.95 / hora
20h: US$ 94.26 / hora
6h: US$ 93.88 / hora
19h: US$ 91.74 / hora
7h: US$ 89.46 / hora
18h: US$ 87.69 / hora
10h: US$ 86.74 / hora


--- TOP 15 LOCAIS DE EMBARQUE MAIS RENTÁVEIS (US$/Hora) ---
Newark Airport (EWR): US$ 139.87/hora
New Dorp/Midland Beach (Staten Island): US$ 136.42/hora
Saint Michaels Cemetery/Woodside (Queens): US$ 129.6/hora
Eltingville/Annadale/Prince's Bay (Staten Island): US$ 125.47/hora
Rikers Island (Bronx): US$ 121.9/hora
LaGuardia Airport (Queens): US$ 118.77/hora
East Elmhurst (Queens): US$ 117.46/hora
JFK Airport (Queens): US$ 117.11/hora
Willets Point (Queens): US$ 116.9/hora
Flushing Meadows-Corona Park (Queens): US$ 108.49/hora
Arrochar/Fort Wadsworth (Staten Island): US$ 106.59/hora
Outside of NYC (nan): US$ 99

In [ ]:
# 1. Criar a regra para classificar a origem
# Se a palavra 'Airport' estiver no nome da Zona, é Aeroporto. Se o Bairro for Manhattan, é Manhattan.
condicoes = [
    df_clean['Zone'].str.contains('Airport', na=False, case=False),
    df_clean['Borough'] == 'Manhattan'
]
escolhas = ['Aeroporto', 'Manhattan']
df_clean['Origem_Agrupada'] = np.select(condicoes, escolhas, default='Outros')

# 2. Filtrar apenas Aeroportos e Manhattan e calcular as médias
df_h1 = df_clean[df_clean['Origem_Agrupada'].isin(['Aeroporto', 'Manhattan'])]
df_medias_h1 = df_h1.groupby('Origem_Agrupada')[['dolares_por_hora', 'fare_amount']].mean().reset_index()

# 3. Criar o gráfico
fig1 = px.bar(
    df_medias_h1, 
    x='Origem_Agrupada', 
    y='dolares_por_hora',
    title='Rentabilidade: Aeroportos vs Manhattan (Dólares por Hora)',
    labels={'Origem_Agrupada': 'Local de Origem', 'dolares_por_hora': 'Média de US$ / Hora'},
    color='Origem_Agrupada',
    text_auto='.2f' # Mostra o valor em cima da barra com 2 casas decimais
)
fig1.show()

In [ ]:
# 1. Calcular a % da gorjeta em relação à tarifa base
df_clean['perc_gorjeta'] = (df_clean['tip_amount'] / df_clean['fare_amount']) * 100

# 2. Agrupar por tipo de pagamento (1=Cartão, 2=Dinheiro) e calcular a média
df_h2 = df_clean.groupby('payment_type')['perc_gorjeta'].mean().reset_index()

# Substituir os números pelos nomes para o gráfico ficar bonito
df_h2['payment_type'] = df_h2['payment_type'].map({1: 'Cartão de Crédito', 2: 'Dinheiro'})

# 3. Criar o gráfico
fig2 = px.bar(
    df_h2, 
    x='payment_type', 
    y='perc_gorjeta',
    title='Média de Gorjeta (%) por Método de Pagamento',
    labels={'payment_type': 'Método de Pagamento', 'perc_gorjeta': '% Média de Gorjeta'},
    color='payment_type',
    text_auto='.2f'
)
fig2.show()

,VendorID,tpep_pickup_datetime,tpep_dropoff_datetime,passenger_count,trip_distance,RatecodeID,store_and_fwd_flag,PULocationID,DOLocationID,payment_type,fare_amount,extra,mta_tax,tip_amount,tolls_amount,improvement_surcharge,total_amount,congestion_surcharge,Airport_fee,cbd_congestion_fee,trip_duration_min,pickup_hour,pickup_day_of_week,tip_percentage,profit_per_mile,dinheiro_recebido,dolares_por_hora
713749,2,2026-01-09 20:14:56,2026-01-09 20:21:52,1.0,295.99,1.0,N,265,229,1,10.7,1.0,0.5,3.29,0.0,1.0,19.74,2.5,0.0,0.75,6.933333,20,Friday,30.747664,0.047265,13.99,121.067308
2466810,1,2026-01-30 09:48:05,2026-01-30 10:11:52,1.0,223.60,99.0,N,11,14,1,25.5,0.0,0.5,0.00,0.0,0.0,26.00,0.0,0.0,0.00,23.783333,9,Friday,0.000000,0.114043,25.50,64.330764
565542,1,2026-01-08 11:25:05,2026-01-08 11:56:17,1.0,192.80,99.0,N,76,117,1,33.5,0.0,0.5,0.00,5.6,0.0,39.60,0.0,0.0,0.00,31.200000,11,Thursday,0.000000,0.173755,33.50,64.423077
2471143,1,2026-01-30 10:55:44,2026-01-30 12:52:24,1.0,157.30,99.0,N,76,49,1,26.5,0.0,0.5,0.00,0.0,0.0,27.00,0.0,0.0,0.00,116.666667,10,Friday,0.000000,0.168468,26.50,13.628571
916965,1,2026-01-12 07:30:36,2026-01-12 08:21:08,1.0,151.60,99.0,N,159,45,1,36.5,0.0,0.5,0.00,0.0,0.0,37.00,0.0,0.0,0.00,50.533333,7,Monday,0.000000,0.240765,36.50,43.337731


In [ ]:
# 1. Filtrar bairros fora de Manhattan (e remover zonas desconhecidas)
bairros_fora = ['Brooklyn', 'Queens', 'Bronx', 'Staten Island']
df_fora = df_clean[df_clean['Borough'].isin(bairros_fora)]

# 2. Contar quantas viagens aconteceram em cada hora, por bairro
df_pico = df_fora.groupby(['Borough', 'pickup_hour']).size().reset_index(name='total_viagens')

# 3. Criar o gráfico de linhas
fig3 = px.line(
    df_pico, 
    x='pickup_hour', 
    y='total_viagens', 
    color='Borough',
    title='Volume de Corridas por Horário (Fora de Manhattan)',
    labels={'pickup_hour': 'Hora do Dia (0-23h)', 'total_viagens': 'Total de Viagens', 'Borough': 'Bairro'},
    markers=True # Coloca pontinhos em cada hora para facilitar a leitura
)
fig3.update_xaxes(dtick=1) # Força o eixo X a mostrar todas as horas de 1 em 1
fig3.show()

In [ ]:
# 1. Criar coluna de Verdadeiro/Falso para gorjetas acima de 20%
df_clean['gorjeta_acima_20'] = df_clean['perc_gorjeta'] > 20

# 2. Criar as faixas de distância (0-2 milhas, 2-5, 5-10, e 10+)
faixas = [0, 2, 5, 10, 100] # O 100 é só um limite superior alto
nomes_faixas = ['0 a 2 milhas', '2 a 5 milhas', '5 a 10 milhas', 'Mais de 10 milhas']
df_clean['faixa_distancia'] = pd.cut(df_clean['trip_distance'], bins=faixas, labels=nomes_faixas)

# 3. Calcular a probabilidade (que é a média de Verdadeiros) multiplicada por 100 para virar %
df_prob = df_clean.groupby('faixa_distancia')['gorjeta_acima_20'].mean().reset_index()
df_prob['gorjeta_acima_20'] = df_prob['gorjeta_acima_20'] * 100

# 4. Criar o gráfico
fig4 = px.bar(
    df_prob,
    x='faixa_distancia',
    y='gorjeta_acima_20',
    title='Probabilidade de Receber Gorjeta > 20% por Distância da Corrida',
    labels={'faixa_distancia': 'Distância da Viagem', 'gorjeta_acima_20': 'Probabilidade (%)'},
    text_auto='.1f',
    color='faixa_distancia'
)
fig4.show()